In [ ]:
import os
import json
import pickle
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import proplot as pplt
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

## 1. Configuration and SR-MED coefficients
Load paths from `configs.json` and SR-MED optimised constants from `optimized_equations.csv`.
The instability pathway is $I = \widehat{\theta_e} - \gamma\,\widehat{\theta_e^*} - \eta$,
where $\gamma$ and $\eta$ are the fitted constants labelled **b** and **c** in the CSV.

In [ ]:
with open('../scripts/configs.json', 'r', encoding='utf-8') as f:
    CONFIGS = json.load(f)

SPLITSDIR  = CONFIGS['filepaths']['splits']
WEIGHTSDIR = CONFIGS['filepaths']['weights']
MODELSDIR  = CONFIGS['filepaths']['models']
MODELS     = CONFIGS['experiments']

SPLIT          = 'test'   # 2018-2020 JJA
NN_SEEDS       = MODELS['nn']['seeds']   # [42, 72, 102]
FIELDVARS      = MODELS['sr']['runs']['sr_gauss']['fieldvars']  # ['rh','thetae','thetaestar']

# --- SR-MED optimised coefficients -------------------------------------------
# Form: a * cube(max(rh, thetae - b*thetaestar - c))
# gamma = b  (weight on theta_e* in the instability pathway)
# eta   = c  (additive offset in the instability pathway)
eqs = pd.read_csv(os.path.join(MODELSDIR, 'sr', 'optimized_equations.csv'))
eqs['constants'] = eqs['constants'].apply(json.loads)
srmed = eqs.set_index('name').loc['sr_med']
GAMMA = srmed['constants']['b']   # 1.4706
ETA   = srmed['constants']['c']   # 0.3756

print(f'SR-MED form : {srmed["form"]}')
print(f'gamma (b)   = {GAMMA}  (theta_e* weight in instability pathway)')
print(f'eta   (c)   = {ETA}   (offset in instability pathway)')

## 2. Load normalised test-period field profiles
Open `norm_test.h5`: the three field variables (`rh`, `thetae`, `thetaestar`) are stored as
5-D arrays `(time, lat, lon, sig)` after normalisation (zero-mean, unit-variance across
training years).  We also need `dsig` (sigma-layer thickness) for vertical integration.

In [ ]:
normpath = os.path.join(SPLITSDIR, f'norm_{SPLIT}.h5')

with xr.open_dataset(normpath, engine='h5netcdf') as ds:
    ntime = ds.sizes['time']
    nlat  = ds.sizes['lat']
    nlon  = ds.sizes['lon']
    nsig  = ds.sizes.get('sig', 1)

    lats = ds['lat'].values          # (nlat,)
    lons = ds['lon'].values          # (nlon,)
    dsig = ds['dsig'].values         # (nsig,) sigma-layer thicknesses, sum to ~0.5
    sigcoords = (ds.coords['sig'].values
                 if 'sig' in ds.coords
                 else np.linspace(0.5, 1.0, nsig))

    # Stack to (ntime*nlat*nlon, nfieldvars, nsig) for bulk kernel integration
    farrs = [
        ds[v].transpose('time', 'lat', 'lon', 'sig').values.reshape(-1, nsig)
        for v in FIELDVARS
    ]
    fieldstack = np.stack(farrs, axis=1)  # (N, 3, nsig)

    # Surface mask (marks levels below surface; None if absent)
    surfmask = (
        ds['surfmask'].transpose('time', 'lat', 'lon', 'sig').values.reshape(-1, nsig)
        if 'surfmask' in ds else None
    )

refshape = (ntime, nlat, nlon)
N = ntime * nlat * nlon

print(f'Domain      : lat {lats.min():.1f}–{lats.max():.1f} N,  '
      f'lon {lons.min():.1f}–{lons.max():.1f} E')
print(f'Grid        : {nlat} lat × {nlon} lon,  {nsig} sigma levels')
print(f'Time steps  : {ntime}  (test = 2018–2020 JJA)')
print(f'Field vars  : {FIELDVARS}')
print(f'fieldstack  : {fieldstack.shape}  (N, nfieldvars, nsig)')
print(f'dsig        : {dsig.round(4)}  (sum = {dsig.sum():.4f})')

## 3. Load Gaussian kernel weights
The `nn_gauss` model learns a separate Gaussian kernel $k(\sigma)$ for each field variable.
Weights are saved per random seed; we average across seeds before integrating.

In [ ]:
kwlist = []
for seed in NN_SEEDS:
    wpath = os.path.join(WEIGHTSDIR, f'nn_gauss_{seed}_weights.nc')
    with xr.open_dataset(wpath, engine='h5netcdf') as wds:
        kw = wds['k'].values          # shape: (nfieldvars, nsig)
        kwlist.append(kw)
        if 'sig' in wds.coords and len(kwlist) == 1:
            sigcoords = wds.coords['sig'].values
        if len(kwlist) == 1:
            field_coords = (wds.coords['field'].values
                            if 'field' in wds.coords else FIELDVARS)

kw_mean = np.mean(kwlist, axis=0)  # (nfieldvars, nsig), averaged over seeds

print(f'Loaded kernel weights for {len(kwlist)} seeds: {NN_SEEDS}')
print(f'Weight array shape : {kw_mean.shape}  (nfieldvars, nsig)')
print(f'Field order        : {field_coords}')
print()
for i, vname in enumerate(field_coords):
    pk = sigcoords[np.argmax(kw_mean[i])]
    print(f'  {vname:12s}  peak sigma={pk:.3f},  '
          f'max weight={kw_mean[i].max():.3f}')

## 4. Kernel integration and pathway diagnostics
The kernel-integrated scalar for each field is
$$\hat{x} = \sum_\sigma k_x(\sigma)\,x(\sigma)\,\Delta\sigma$$
yielding $\widehat{\mathrm{RH}}$, $\widehat{\theta_e}$, and $\widehat{\theta_e^*}$ at every grid point and time step.

We then compute the two SR-MED pathways:
$$M = \widehat{\mathrm{RH}}, \qquad
  I = \widehat{\theta_e} - \gamma\,\widehat{\theta_e^*} - \eta$$
and their difference $M - I$ (positive where moisture-dominated, negative where instability-dominated).

In [ ]:
def kernel_integrate(fields, weights, dsig, mask=None):
    """Weighted vertical integral.  fields: (N, nfieldvars, nsig)."""
    w = fields * weights[None, :, :] * dsig[None, None, :]
    if mask is not None:
        w = w * mask[:, None, :]
    return w.sum(axis=2)   # (N, nfieldvars)

# Average kernel-integrated features across seeds
ki_per_seed = [
    kernel_integrate(fieldstack, kw, dsig, surfmask)
    for kw in kwlist
]
ki = np.mean(ki_per_seed, axis=0)   # (N, 3)

# Unpack into named 1-D arrays (length N = ntime*nlat*nlon)
M_flat   = ki[:, 0]                              # RH-hat
te_flat  = ki[:, 1]                              # theta_e-hat
tes_flat = ki[:, 2]                              # theta_e_star-hat
I_flat   = te_flat - GAMMA * tes_flat - ETA      # instability pathway
MI_flat  = M_flat - I_flat                       # positive => moisture-dominated

# Quick sanity check
frac_moist = np.mean(M_flat >= I_flat)
print(f'Kernel-integrated features computed.')
print(f'  M_flat   : mean={M_flat.mean():.4f},  std={M_flat.std():.4f}')
print(f'  te_flat  : mean={te_flat.mean():.4f},  std={te_flat.std():.4f}')
print(f'  tes_flat : mean={tes_flat.mean():.4f},  std={tes_flat.std():.4f}')
print(f'  I_flat   : mean={I_flat.mean():.4f},  std={I_flat.std():.4f}')
print(f'  MI_flat  : mean={MI_flat.mean():.4f},  std={MI_flat.std():.4f}')
print(f'Fraction moisture-dominated (M >= I): {100*frac_moist:.1f}%')

## 5. Reshape to spatial grids and time-average
Reshape each flat array from `(N,)` to `(ntime, nlat, nlon)`, then take the
JJA mean over the 2018–2020 test period to get a single `(nlat, nlon)` map.

In [ ]:
def to_map(arr, refshape):
    return arr.reshape(refshape).mean(axis=0)

M_map   = to_map(M_flat,   refshape)   # RH-hat, time-mean
te_map  = to_map(te_flat,  refshape)   # theta_e-hat, time-mean
tes_map = to_map(tes_flat, refshape)   # theta_e_star-hat, time-mean
I_map   = to_map(I_flat,   refshape)   # instability pathway, time-mean
MI_map  = to_map(MI_flat,  refshape)   # M - I, time-mean

print(f'Spatial maps computed: each shape = {M_map.shape}  (nlat × nlon)')
print()
for label, arr in [('M   (RH-hat)', M_map),
                   ('te  (te-hat)', te_map),
                   ('tes (tes-hat)', tes_map),
                   ('I            ', I_map),
                   ('M - I        ', MI_map)]:
    print(f'  {label}:  min={np.nanmin(arr):.3f},  '
          f'mean={np.nanmean(arr):.3f},  '
          f'max={np.nanmax(arr):.3f}')

## 6. Five-panel spatial map
Panels 1–4 use a diverging (`RdBu_r`) colormap with symmetric limits, sharing a
single colorbar.  Panel 5 uses `ColdHot_r` (blue = moisture-dominated, red =
instability-dominated) with a contour marking the $M = I$ transition.

In [ ]:
lonlim = (float(lons.min()), float(lons.max()))
latlim = (float(lats.min()), float(lats.max()))

# --- colour limits -----------------------------------------------------------
# Panels 1–4: symmetric, based on pooled 98th-percentile absolute value
pool14 = np.concatenate([arr.ravel() for arr in [M_map, te_map, tes_map, I_map]
                         if arr is not None])
vmax14 = float(np.nanpercentile(np.abs(pool14), 98))

# Panel 5: symmetric around zero
vmax5  = float(np.nanpercentile(np.abs(MI_map.ravel()), 98))

# Common map format kwargs
kwmap = dict(
    coast=True, coastlinewidth=0.5,
    lonlim=lonlim, latlim=latlim,
    lonlines=10, latlines=10,
    lonlabels='b', latlabels='l',
    grid=False,
)

# pcolormesh kwargs for panels 1–4
kw14 = dict(
    cmap='RdBu_r',
    vmin=-vmax14, vmax=vmax14,
    levels=15,
)

# pcolormesh kwargs for panel 5 (diverging centred at zero)
kw5 = dict(
    cmap='ColdHot_r',
    vmin=-vmax5, vmax=vmax5,
    levels=15,
)

# --- layout ------------------------------------------------------------------
# [[1,2,3],[0,4,5]]  →  top row: M, te, tes;  bottom row (centred): I, M-I
fig, axs = pplt.subplots(
    [[1, 2, 3], [0, 4, 5]],
    proj='cyl',
    figwidth=7.0,
    hspace=2.5,
    wspace=1.5,
)

# panels 1–4 share a colormap; collect one handle for the shared colorbar
panels14 = []

# Panel 1: M = RH-hat
m0 = axs[0].pcolormesh(lons, lats, M_map, **kw14)
axs[0].format(title=r'$\widehat{\mathrm{RH}}\;(M)$', **kwmap)
panels14.append(m0)

# Panel 2: theta_e-hat
m1 = axs[1].pcolormesh(lons, lats, te_map, **kw14)
axs[1].format(title=r'$\widehat{\theta_e}$', **kwmap)
panels14.append(m1)

# Panel 3: theta_e_star-hat
m2 = axs[2].pcolormesh(lons, lats, tes_map, **kw14)
axs[2].format(title=r'$\widehat{\theta_e^*}$', **kwmap)
panels14.append(m2)

# Panel 4: I = te-hat - gamma * tes-hat - eta
m3 = axs[3].pcolormesh(lons, lats, I_map, **kw14)
axs[3].format(
    title=r'$I = \widehat{\theta_e} - \gamma\widehat{\theta_e^*} - \eta$',
    **kwmap,
)
panels14.append(m3)

# Shared colorbar for panels 1–4
fig.colorbar(
    m0,
    loc='b', span=[1, 3],
    label='Normalised pathway value (std. dev.)',
    ticks=pplt.arange(-vmax14, vmax14, vmax14 / 2),
)

# Panel 5: M - I  (diverging, centred at zero)
m4 = axs[4].pcolormesh(lons, lats, MI_map, **kw5)

# M = I transition contour (zero line of M - I)
axs[4].contour(
    lons, lats, MI_map,
    levels=[0.0],
    colors='k', linewidths=0.8, linestyles='-',
)
axs[4].format(title=r'$M - I$', **kwmap)
axs[4].colorbar(
    m4,
    loc='b',
    label=r'$M - I$ (std. dev.)',
)

fig.format(abc=True, abcloc='ul', titleloc='l')
pplt.show()
fig.save('../figs/fig_pathway_maps.jpg')